In [5]:
import os
import faiss
import torch
import numpy as np

# Force Hugging Face libraries to stay offline
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_EVALUATE_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

### Sanity Check for FAISS Database (.index)

In [2]:
INDEX_PATH = "./pii_audit_target.index"

# 1. Basic file check
print("Exists:", os.path.exists(INDEX_PATH))
print("File size (bytes):", os.path.getsize(INDEX_PATH) if os.path.exists(INDEX_PATH) else "missing")

# 2. Load the FAISS index
index = faiss.read_index(INDEX_PATH)

# 3. Structural sanity checks
print("ntotal:", index.ntotal)   # number of vectors
print("dimension:", index.d)     # vector dimension
print("is_trained:", index.is_trained)

# Expected values for your setup
expected_ntotal = 43000   # or 5000 if you only indexed 5k
expected_dim = 768

print("ntotal correct:", index.ntotal == expected_ntotal)
print("dimension correct:", index.d == expected_dim)

Exists: True
File size (bytes): 132096045
ntotal: 43000
dimension: 768
is_trained: True
ntotal correct: True
dimension correct: True


### Search the Index with Known Vectors

In [3]:
vectors = np.load("embeddings.npy")

In [4]:
# Search a few known vectors against the index
k = 5
query_ids = [0, 1, 2, 10, 100]

for qid in query_ids:
    query = vectors[qid:qid+1].astype("float32")
    distances, indices = index.search(query, k)

    print(f"\nQuery vector {qid}")
    print("Nearest neighbor indices:", indices[0])
    print("Distances:", distances[0])
    print("Self-match correct:", indices[0][0] == qid)


Query vector 0
Nearest neighbor indices: [    0 38332 31322 42523  3959]
Distances: [5.1468920e-13 3.5805452e-01 3.6613363e-01 3.6907005e-01 3.8634893e-01]
Self-match correct: True

Query vector 1
Nearest neighbor indices: [    1 24166 19866  3731 32601]
Distances: [2.317728e-13 7.010814e-01 7.013104e-01 7.277334e-01 7.283392e-01]
Self-match correct: True

Query vector 2
Nearest neighbor indices: [    2 12901 34109 41347 34142]
Distances: [2.2400223e-13 6.3987541e-01 6.7237318e-01 6.7811418e-01 6.9281590e-01]
Self-match correct: True

Query vector 10
Nearest neighbor indices: [   10 33440 12019 39684 18380]
Distances: [2.6093271e-13 6.5332496e-01 6.8383563e-01 6.8978554e-01 6.9279325e-01]
Self-match correct: True

Query vector 100
Nearest neighbor indices: [  100 33960 31604 29264  9705]
Distances: [2.8893118e-13 5.5970359e-01 6.2158948e-01 6.4041418e-01 6.5757990e-01]
Self-match correct: True


For each query vector, FAISS returns the 5 nearest vectors in the index.

For example:

* `Nearest neighbor indices: [0, 38332, 31322, 42523, 3959]`

* The first result is 0, which means vector 0’s closest match is itself.

That is exactly what we want when we search using a vector that is already inside the index.

##### How to read each part

`Query vector 0`
We searched using the embedding at row 0.

`Nearest neighbor indices: [0 38332 31322 42523 3959]`
These are the row IDs of the closest vectors in the FAISS index, ordered from most similar to less similar.

`0` is the closest

`38332` is the second closest

and so on

`Distances: [5.1468920e-13 3.5805452e-01 3.6613363e-01 ...]`
These are the L2 distances because we used:

### Compare Custom Text to Database Embeddings

In [6]:
import pickle

if os.path.exists("processed_pii_texts.pkl"):
    with open("processed_pii_texts.pkl", "rb") as f:
        texts_for_db = pickle.load(f)
    print(f"Ready! Loaded {len(texts_for_db)} pre-processed PII samples.")

Ready! Loaded 43000 pre-processed PII samples.


In [8]:
from transformers import AutoTokenizer, T5EncoderModel

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_LOCAL_PATH = "./gtr-t5-base-local"

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_LOCAL_PATH)

# Load encoder-only model
model = T5EncoderModel.from_pretrained(MODEL_LOCAL_PATH)
model.to(device)
model.eval()

/home/pmd4nd/.conda/envs/privacy_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 99/99 [00:00<00:00, 22694.22it/s]


T5EncoderModel(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dropout(p=0.1, 

In [9]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_text(text):
    inputs = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        embedding = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])

    return embedding.cpu().numpy().astype("float32")

In [12]:
custom_text = "Omer, the license 78B5R2MVFAHJ48500 still appears in our database as being registered for access."

query_vec = embed_text(custom_text)
distances, indices = index.search(query_vec, 5)

print("Top-5 neighbor indices:", indices[0])
print("Top-5 distances:", distances[0])

# Uncomment code below to use cosine similarity instead
# query_vec = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)

# faiss.normalize_L2(query_vec)

# D, I = index.search(query_vec, 5)

# print("Nearest neighbors:", I[0])
# print("Cosine similarity scores:", D[0])

Top-5 neighbor indices: [    1 32601  4476 13115  8740]
Top-5 distances: [0.20968519 0.6859009  0.70199347 0.7103317  0.71385384]


In [ ]:
query_vec = embed_text(custom_text)

In [11]:
for rank, idx in enumerate(indices[0]):
    print(f"\nRank {rank+1}")
    print("Index:", idx)
    print("Distance:", distances[0][rank])
    print("Text:", texts_for_db[idx])


Rank 1
Index: 1
Distance: 0.20968519
Text: Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access

Rank 2
Index: 32601
Distance: 0.6859009
Text: Kindly present your 3HTCFGHGM1H056477 registration and 1NXWTXGTL8AsHK5qN

Rank 3
Index: 4476
Distance: 0.70199347
Text: The identification number 41-427752-827137-6 logged into the patient portal from location [28.9231,176.5321]

Rank 4
Index: 13115
Distance: 0.7103317
Text: Pay occupational safety licensing fees via CY9563001006647277422T50V222. Include your KJ7YG7D

Rank 5
Index: 8740
Distance: 0.71385384
Text: Patient 0x6ca9acab8506ec52a77bc3ac8bdf9be405ee
